In [1]:
import sys
import pickle
sys.path.append('../../TaskExecutionTimeMining/')
from divide_and_conquer import *
from sklearn.feature_selection import mutual_info_regression

import numpy as np
np.seterr(divide='ignore', invalid='ignore')

{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

In [ ]:
with open("../transformed_event_logs/BPIC_2017_all_train.pickle", "rb") as f:
    event_log = pickle.load(f)

# numerical attributes : duration, seconds_in_day, day_in_week
numerical_attributes = [
    'duration_seconds',
    'seconds_in_day',
    #'day_of_week',
    'case:RequestedAmount_start'
]

transformed_event_log = event_log.copy()

for num_attr in numerical_attributes:
    transformed_event_log[num_attr] = np.log(transformed_event_log[num_attr]+1)
    transformed_event_log[num_attr] = (transformed_event_log[num_attr] - transformed_event_log[num_attr].mean()) / transformed_event_log[num_attr].std()

/tmp/ipykernel_147202/2135300804.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  event_log = pickle.load(f)


In [3]:
target_column = 'duration_seconds'
continuous_feature_columns = ['seconds_in_day']
nominal_feature_columns = ['day_of_week', 'concept:name', 'org:resource_start']

In [4]:
mi_matrix = calculate_mi_matrix(transformed_event_log, target_column, continuous_feature_columns, nominal_feature_columns,
                                verbose=True)

Computing MI:   0%|          | 0/14 [00:00<?, ?it/s]

MI(day_of_week, day_of_week) = 2.5719929814033864
MI(org:resource_start, org:resource_start) = 6.476911727822364
MI(concept:name, concept:name) = 4.2632097093313694
MI(day_of_week, org:resource_start) = 0.15239403038941957
MI(day_of_week, concept:name) = 0.05212193695465015
MI(concept:name, org:resource_start) = 0.9226588415808465


MI(seconds_in_day, seconds_in_day) = 1.9904924143308262


MI(seconds_in_day, duration_seconds) = 2.4942439320808445


MI(day_of_week, seconds_in_day) = 0.07194532654659033


MI(day_of_week, duration_seconds) = 0.15615412393851946


MI(concept:name, seconds_in_day) = 0.1438874688629333


MI(concept:name, duration_seconds) = 1.2552724197286234


MI(org:resource_start, seconds_in_day) = 0.25189600744364005


MI(org:resource_start, duration_seconds) = 0.3355914588304512


In [5]:
mimr, all_relevance = calculate_maximal_relevance_minimal_redundancy_split(mi_matrix, target_column, continuous_feature_columns, nominal_feature_columns)
print(mimr)
print(all_relevance)

seconds_in_day
{'day_of_week': np.float64(-0.555959444884992), 'concept:name': np.float64(-0.09019706945382677), 'org:resource_start': np.float64(-1.6153736929786162), 'seconds_in_day': np.float64(1.879688627784847)}
